In [19]:
import requests
import pandas as pd
import psycopg2
from pathlib import Path
import os
from dotenv import load_dotenv

In [20]:
load_dotenv(override=True)
url = os.getenv("URL")
file_path = os.getenv("FILE_PATH")

conn = psycopg2.connect(
    database=os.getenv("DB_NAME"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASS"),
    host=os.getenv("DB_HOST"),
    port=os.getenv("DB_PORT")
)

cursor = conn.cursor()

original_nyc_data = None
if not os.path.exists(file_path):
    req = requests.get(url)
    original_nyc_data = pd.DataFrame(data = req.json())
    output_file = Path(file_path)
    output_file.parent.mkdir(exist_ok=True, parents=True)
    original_nyc_data.to_csv(file_path,encoding='utf-8', index=False)
else:
    original_nyc_data = pd.read_csv(file_path)


In [21]:
copied_nyc_arrest = original_nyc_data.copy()

In [22]:
copied_nyc_arrest.head()

,arrest_key,arrest_date,pd_cd,pd_desc,ky_cd,ofns_desc,law_code,law_cat_cd,arrest_boro,arrest_precinct,jurisdiction_code,age_group,perp_sex,perp_race,x_coord_cd,y_coord_cd,latitude,longitude,lon_lat
0,279197226,2023-12-19T00:00:00.000,105.0,STRANGULATION 1ST,106.0,FELONY ASSAULT,PL 1211200,F,M,18,0,25-44,M,WHITE,988210.0,218129.0,40.765390,-73.985702,"{'type': 'Point', 'coordinates': [-73.985702, ..."
1,278761840,2023-12-09T00:00:00.000,105.0,STRANGULATION 1ST,106.0,FELONY ASSAULT,PL 1211300,F,K,67,0,25-44,M,BLACK,997897.0,175676.0,40.648859,-73.950820,"{'type': 'Point', 'coordinates': [-73.95082, 4..."
2,278506761,2023-12-05T00:00:00.000,153.0,RAPE 3,104.0,RAPE,PL 1302503,F,K,77,0,25-44,M,BLACK,1003509.0,185018.0,40.674496,-73.930571,"{'type': 'Point', 'coordinates': [-73.93057132..."
3,278436408,2023-12-03T00:00:00.000,157.0,RAPE 1,104.0,RAPE,PL 1303501,F,B,46,0,45-64,M,BLACK,1011755.0,250279.0,40.853598,-73.900577,"{'type': 'Point', 'coordinates': [-73.90057688..."
4,278248753,2023-11-29T00:00:00.000,660.0,(null),NaN,(null),PL 2407800,M,Q,104,0,<18,M,WHITE HISPANIC,1011456.0,194092.0,40.699373,-73.901881,"{'type': 'Point', 'coordinates': [-73.901881, ..."


In [23]:
copied_nyc_arrest['arrest_date'] = pd.to_datetime(copied_nyc_arrest['arrest_date'])

In [24]:
copied_nyc_arrest['arrest_date'].head()

0   2023-12-19
1   2023-12-09
2   2023-12-05
3   2023-12-03
4   2023-11-29
Name: arrest_date, dtype: datetime64[ns]

In [25]:
# replace null pd_desc, ky_cd and offns_desc with unknown
copied_nyc_arrest[copied_nyc_arrest['pd_desc'] == "(null)"].head()

,arrest_key,arrest_date,pd_cd,pd_desc,ky_cd,ofns_desc,law_code,law_cat_cd,arrest_boro,arrest_precinct,jurisdiction_code,age_group,perp_sex,perp_race,x_coord_cd,y_coord_cd,latitude,longitude,lon_lat
4,278248753,2023-11-29,660.0,(null),NaN,(null),PL 2407800,M,Q,104,0,<18,M,WHITE HISPANIC,1011456.0,194092.0,40.699373,-73.901881,"{'type': 'Point', 'coordinates': [-73.901881, ..."
30,266655043,2023-04-13,777.0,(null),NaN,(null),PL 1950200,F,K,81,0,45-64,M,BLACK,1005312.0,190540.0,40.689640,-73.924051,"{'type': 'Point', 'coordinates': [-73.924051, ..."
43,260331614,2022-12-21,NaN,(null),NaN,(null),CPL5700600,9,Q,103,0,18-24,M,BLACK,1040435.0,195998.0,40.704469,-73.797355,"{'type': 'Point', 'coordinates': [-73.797355, ..."
44,257876691,2022-12-19,777.0,(null),NaN,(null),PL 1950200,F,Q,105,0,45-64,M,WHITE HISPANIC,1057766.0,203992.0,40.726284,-73.734760,"{'type': 'Point', 'coordinates': [-73.73476, 4..."
46,256493373,2022-12-17,NaN,(null),NaN,(null),CPL5700600,9,Q,102,0,25-44,M,WHITE,1032501.0,198800.0,40.712206,-73.825952,"{'type': 'Point', 'coordinates': [-73.825952, ..."


In [26]:
copied_nyc_arrest = copied_nyc_arrest.replace({"pd_desc": {"(null)": "Unknown"}})
copied_nyc_arrest = copied_nyc_arrest.fillna({"pd_desc": "unknown"})

In [27]:
copied_nyc_arrest['pd_desc'].isna().sum()

np.int64(0)

In [28]:
copied_nyc_arrest.loc[copied_nyc_arrest['pd_desc'] == '(null)'].shape[0]

0

In [29]:
copied_nyc_arrest.drop(columns=["ky_cd", "ofns_desc", "lon_lat"], inplace=True)

In [30]:
copied_nyc_arrest.columns

Index(['arrest_key', 'arrest_date', 'pd_cd', 'pd_desc', 'law_code',
       'law_cat_cd', 'arrest_boro', 'arrest_precinct', 'jurisdiction_code',
       'age_group', 'perp_sex', 'perp_race', 'x_coord_cd', 'y_coord_cd',
       'latitude', 'longitude'],
      dtype='object')

In [31]:
cursor.execute("DROP TABLE IF EXISTS nyc_crime_staging")
conn.commit()

In [32]:
cursor.execute("""CREATE TABLE IF NOT EXISTS nyc_crime_staging
               (
                   arrest_key TEXT PRIMARY KEY,
                   arrest_date DATE,
                   pd_cd NUMERIC,
                   pd_desc VARCHAR(255),
                   law_code VARCHAR(255),
                   law_cat_cd VARCHAR(50),
                   arrest_boro VARCHAR(10),
                   arrest_precinct NUMERIC,
                   jurisdiction_code NUMERIC,
                   age_group VARCHAR(50),
                   perp_sex VARCHAR(20),
                   perp_race VARCHAR(100),
                   x_coord_cd TEXT,
                   y_coord_cd TEXT,
                   latitude NUMERIC,
                   longitude NUMERIC 
               )
               """
               )
conn.commit()

In [33]:
output_file = Path(os.getenv("STAGING_PATH"))
output_file.parent.mkdir(exist_ok=True, parents=True)
copied_nyc_arrest.to_csv(os.getenv("STAGING_PATH"), encoding='utf-8', index=False)

In [34]:
# 278254593,2023-11-29,464.0,JOSTLING,PL 1652501,M,M,18,0,<18,M,WHITE HISPANIC,990503.0,215519.0,40.758225,-73.977428
with open(os.getenv("STAGING_PATH"), 'r', encoding='utf-8') as f:
    next(f)
    cursor.copy_expert(
        """
        COPY nyc_crime_staging 
        FROM STDIN 
        WITH (FORMAT CSV, HEADER TRUE, DELIMITER ',');            
        """, 
        f)

conn.commit()

In [35]:
cursor.execute("""CREATE TABLE IF NOT EXISTS nyc_crime_cleaned
               (
                   arrest_key TEXT PRIMARY KEY,
                   arrest_date DATE,
                   pd_cd NUMERIC,
                   pd_desc VARCHAR(255),
                   law_code VARCHAR(255),
                   law_cat_cd VARCHAR(50),
                   arrest_boro VARCHAR(10),
                   arrest_precinct NUMERIC,
                   jurisdiction_code NUMERIC,
                   age_group VARCHAR(50),
                   perp_sex VARCHAR(20),
                   perp_race VARCHAR(100),
                   x_coord_cd TEXT,
                   y_coord_cd TEXT,
                   latitude NUMERIC,
                   longitude NUMERIC 
               )
               """
               )
conn.commit()

In [36]:
cursor.execute(
    """
    INSERT INTO nyc_crime_cleaned (
        arrest_key, arrest_date, pd_cd, pd_desc, law_code, 
        law_cat_cd, arrest_boro, arrest_precinct, jurisdiction_code, 
        age_group, perp_sex, perp_race, x_coord_cd, y_coord_cd, 
        latitude, longitude
    )
    SELECT 
        arrest_key, arrest_date, pd_cd, pd_desc, law_code, 
        law_cat_cd, arrest_boro, arrest_precinct, jurisdiction_code, 
        age_group, perp_sex, perp_race, x_coord_cd, y_coord_cd, 
        latitude, longitude
    FROM nyc_crime_staging
    ON CONFLICT (arrest_key) 
    DO UPDATE SET 
        arrest_date = EXCLUDED.arrest_date,
        pd_cd = EXCLUDED.pd_cd,
        pd_desc = EXCLUDED.pd_desc,
        law_code = EXCLUDED.law_code,
        law_cat_cd = EXCLUDED.law_cat_cd,
        arrest_boro = EXCLUDED.arrest_boro,
        arrest_precinct = EXCLUDED.arrest_precinct,
        jurisdiction_code = EXCLUDED.jurisdiction_code,
        age_group = EXCLUDED.age_group,
        perp_sex = EXCLUDED.perp_sex,
        perp_race = EXCLUDED.perp_race,
        x_coord_cd = EXCLUDED.x_coord_cd,
        y_coord_cd = EXCLUDED.y_coord_cd,
        latitude = EXCLUDED.latitude,
        longitude = EXCLUDED.longitude;
    """
)

conn.commit()